In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_path = "/Volumes/azureproject/dbo/bronze/"
silver_path = "/Volumes/azureproject/dbo/silver/"


In [0]:
calendar              = "AdventureWorks_Calendar"
customers             = "AdventureWorks_Customers"
product_categories    = "AdventureWorks_Product_Categories"
products              = "AdventureWorks_Products"
returns               = "AdventureWorks_Returns"
sales_2015            = "AdventureWorks_Sales_2015"
sales_2016            = "AdventureWorks_Sales_2016"
sales_2017            = "AdventureWorks_Sales_2017"
territories           = "AdventureWorks_Territories"
product_subcategories = "Product_Subcategories"


###Reading Data

In [0]:
df_cal = spark.read.csv(bronze_path + calendar, header=True, inferSchema=True)


In [0]:
df_cus = spark.read.csv(bronze_path + customers, header=True, inferSchema=True)

In [0]:
df_procat = spark.read.csv(bronze_path + product_categories, header=True, inferSchema=True)
df_pro = spark.read.csv(bronze_path + products, header=True, inferSchema=True)
df_ret = spark.read.csv(bronze_path + returns, header=True, inferSchema=True)
df_sal15 = spark.read.csv(bronze_path + sales_2015, header=True, inferSchema=True)
df_sal16 = spark.read.csv(bronze_path + sales_2016, header=True, inferSchema=True)
df_sal17 = spark.read.csv(bronze_path + sales_2017, header=True, inferSchema=True)
df_ter = spark.read.csv(bronze_path + territories, header=True, inferSchema=True)
df_prosub = spark.read.csv(bronze_path + product_subcategories, header=True, inferSchema=True)

In [0]:
df_sales = df_sal15.union(df_sal16).union(df_sal17)

### Transformations

In [0]:
df_cal = df_cal.withColumn('month',month(col("Date"))) \
                .withColumn("year",year(col("Date")))


In [0]:
df_cal.write.mode("overwrite").parquet(silver_path + calendar)

In [0]:
df_cus = df_cus.withColumn("FullName",  concat_ws(" ", "Prefix", "FirstName", "LastName"))

In [0]:
df_cal.write.mode("overwrite").parquet(silver_path + customers)

In [0]:
df_prosub.write.mode("overwrite").parquet(silver_path + product_subcategories)

In [0]:
df_pro = df_pro.withColumn("ProductSKU",split(col("ProductSKU"),"-").getItem(0)) \
                .withColumn("ProductName",split(col("ProductName")," ").getItem(0))
    

In [0]:
df_pro.write.mode("overwrite").parquet(silver_path + products)


In [0]:
df_ret.write.mode("overwrite").parquet(silver_path + returns)

In [0]:
df_ter.write.mode("overwrite").parquet(silver_path + territories)

In [0]:
df_sales =df_sales.withColumn("StockDate", to_timestamp(col("StockDate")))

In [0]:
df_sales = df_sales.withColumn("OrderNumber",regexp_replace(col("OrderNumber"),"S","T"))


In [0]:
df_sales = df_sales.withColumn("multiply",col("OrderLineItem") * col("OrderQuantity"))

### Sales Analysis

In [0]:
df_sales.groupBy("OrderDate").agg(count("OrderNumber").alias("total_orders")).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_procat.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_ter.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_sales.write.mode("overwrite").parquet(silver_path + 'AdventureWorks_Sales')

In [0]:
df_procat.write.mode("overwrite").parquet(silver_path + product_categories)